# 02 — Huấn luyện bước 0
Phân loại AI-touched (không phải kết luận đạo văn). Dữ liệu Humarin + PADBen Task 1/2/3.

Notebook gọi script chuẩn trong `src/`, không nhân bản training loop.

## Môi trường và đường dẫn
Chọn kernel Python đã cài dependency của repo. Notebook không tự cài/upgrade PyTorch.
Chạy các ô từ trên xuống. Có thể cấu hình `CATEX_PROJECT_ROOT`, `CATEX_DATA_DIR`, `CATEX_OUTPUT_ROOT` hoặc sửa ô dưới.
Không đưa `.env` lên notebook. Các notebook này không gọi Web Search hay LLM.


In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display

# Local: mở notebook từ repo hoặc notebooks/. Kaggle: sửa PROJECT_ROOT trỏ tới code đã giải nén.
PROJECT_ROOT = os.environ.get("CATEX_PROJECT_ROOT", "")
if PROJECT_ROOT:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/models/runtime.py").is_file()), None)
if ROOT is None or not (ROOT / "src/models/runtime.py").is_file():
    raise FileNotFoundError("Đặt PROJECT_ROOT thành thư mục chứa src/ và notebooks/.")
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

# Kaggle: sửa thành đường dẫn thư mục split hiển thị trong Input; không ghi vào Input.
DATA_DIR = Path(os.environ.get("CATEX_DATA_DIR", str(ROOT / "data/splits"))).resolve()
DEFAULT_OUTPUT = Path("/kaggle/working/catex") if Path("/kaggle").exists() else ROOT
OUTPUT_ROOT = Path(os.environ.get("CATEX_OUTPUT_ROOT", str(DEFAULT_OUTPUT))).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
from notebooks.workflow import inspect_csv, run_script, read_json, require_checkpoint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Code:", ROOT, "\nData:", DATA_DIR, "\nOutput:", OUTPUT_ROOT)


## Cấu hình
Mặc định chỉ kiểm tra cấu hình và in lệnh. Bật `RUN_TRAIN=True` để train, `RUN_TEST=True` để đánh giá checkpoint cố định trên test.
Đã có checkpoint `last` định kỳ và resume giữa epoch. `best` chỉ có sau validation.
Mặc định tự resume `last` trong thư mục hiện tại; phiên Kaggle mới đặt `RESUME_FROM` trỏ tới thư mục checkpoint từ Input.
`MAX_RUNTIME_MINUTES=480` là ngân sách chủ động cho lệnh train, không phải cam kết hạn mức Kaggle; giảm nếu quota còn ít. Chừa thời gian khởi tạo notebook và lưu output.
Không tự giảm dữ liệu hoặc đổi split. Lần đầu nạp backbone có thể tải từ Internet.

In [ ]:
RUN_TRAIN=False
RUN_TEST=False
ALLOW_CPU_TRAIN=False
# Lần đầu để rỗng. Phiên sau: đường dẫn checkpoint đã gắn vào Kaggle Input.
RESUME_FROM=""
AUTO_RESUME=True
SAVE_EVERY_STEPS=500
SAVE_EVERY_MINUTES=10
MAX_RUNTIME_MINUTES=480  # 8 giờ cho lệnh train; None nếu không cần tự dừng
RUN_NAME="baseline_seed42"
SEED=42
BATCH_SIZE=16
EPOCHS=5
MAX_LENGTH=128
LEARNING_RATE=2e-5
PATIENCE=2
from src.models.tang0_style_classifier.model import DEFAULT_PRETRAINED_NAME
BACKBONE=DEFAULT_PRETRAINED_NAME
TRAIN_CSV=DATA_DIR / "tang0_train.csv"
VAL_CSV=DATA_DIR / "tang0_val.csv"
TEST_CSV=DATA_DIR / "tang0_test.csv"
RUN_DIR=OUTPUT_ROOT / "experiments" / f"tang0_{RUN_NAME}"
CHECKPOINT_DIR=RUN_DIR / "checkpoint"
RUN_DIR.mkdir(parents=True,exist_ok=True)


In [ ]:
import torch
print("PyTorch:",torch.__version__,"GPU:",torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
if RUN_TRAIN and not torch.cuda.is_available() and not ALLOW_CPU_TRAIN:
    raise RuntimeError("Chưa có GPU. Bật GPU hoặc đặt ALLOW_CPU_TRAIN=True nếu cố ý chạy CPU.")
for path in (TRAIN_CSV,VAL_CSV,TEST_CSV):
    if not path.is_file():raise FileNotFoundError(path)
    preview=pd.read_csv(path,nrows=8)
    assert set(['sentence']).issubset(preview.columns)
    print(path.name);display(preview.head(2))
if RUN_TRAIN and AUTO_RESUME and not RESUME_FROM and (CHECKPOINT_DIR / "tang0_last.pt").is_file():
    RESUME_FROM=str(CHECKPOINT_DIR)
if RUN_TRAIN and any(CHECKPOINT_DIR.glob("*.pt")) and not RESUME_FROM:
    raise FileExistsError("Đã có model. Đặt RESUME_FROM tới last hoặc đổi RUN_NAME; best cũ không có optimizer.")
if RESUME_FROM:
    resume_path=Path(RESUME_FROM)
    if resume_path.is_dir(): resume_path=resume_path / "tang0_last.pt"
    if not resume_path.is_file(): raise FileNotFoundError(resume_path)
    print("Tiếp tục từ:",resume_path)


In [ ]:
arguments=["--train",TRAIN_CSV,"--val",VAL_CSV,"--output-dir",CHECKPOINT_DIR,
           "--pretrained-name",BACKBONE,"--max-length",MAX_LENGTH,"--batch-size",BATCH_SIZE,
           "--learning-rate",LEARNING_RATE,"--num-epochs",EPOCHS,"--patience",PATIENCE,"--seed",SEED]
arguments += ["--save-every-steps",SAVE_EVERY_STEPS,"--save-every-minutes",SAVE_EVERY_MINUTES]
if MAX_RUNTIME_MINUTES is not None:
    arguments += ["--max-runtime-minutes",MAX_RUNTIME_MINUTES]
if RESUME_FROM:
    arguments += ["--resume-from",RESUME_FROM]
if RUN_TRAIN:
    manifest={"tier":0,"arguments":[str(a) for a in arguments],"torch":torch.__version__,"data_dir":str(DATA_DIR)}
    (RUN_DIR / "run_config.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
run_script(ROOT,"src/models/tang0_style_classifier/train.py",arguments,enabled=RUN_TRAIN,log_path=RUN_DIR / "train.log")


## Lịch sử loss và biểu đồ
CSV được lưu cùng mỗi checkpoint, lịch sử đi theo `last` khi resume sang phiên mới.
`steps_history.csv`: loss từng batch, global step và learning rate. `epochs_history.csv`: train/val loss và metric sau validation.
Loss epoch là trung bình loss các batch (bước 0 dùng cùng class weights từ train cho cả hai tập); batch cuối nhỏ vẫn tính một batch. Train loss đo trong quá trình cập nhật với dropout, val loss đo cuối epoch ở eval mode nên không phải phép so sánh tuyệt đối.
Đường trung bình trượt chỉ dùng hiển thị, CSV giữ loss gốc. Chạy lại ô này bất cứ lúc nào, không cần train lại/GPU.
Checkpoint cũ chưa lưu history thì không thể dựng lại loss từng bước trước đó; phần lịch sử mới bắt đầu từ lúc cập nhật code.
Train loss giảm nhưng val loss tăng kéo dài có thể là dấu hiệu overfitting; xem thêm F1/PR-AUC hoặc ranking accuracy, không kết luận chất lượng chỉ từ loss.

In [ ]:
from notebooks.training_plots import plot_training_history
fig = plot_training_history(CHECKPOINT_DIR, smoothing_window=100)
if fig is not None:
    plt.show()


## Checkpoint và đánh giá test
Ngưỡng/epoch được chọn bằng validation. Không chọn lại ngưỡng trên test. Chỉ bật đánh giá test khi đã chốt cấu hình.

In [ ]:
status=read_json(CHECKPOINT_DIR / "training_status.json")
display(status)
if RUN_TEST and status is not None and not status["finished"]:
    print("Train đang tạm dừng; chưa chạy test. Resume và hoàn tất trước.")
    RUN_TEST=False
metadata=read_json(CHECKPOINT_DIR / "model_config.json")
display(metadata)
if RUN_TEST:require_checkpoint(CHECKPOINT_DIR,0)
run_script(ROOT,"evaluation/evaluate_tang0.py",
    ["--input",TEST_CSV,"--checkpoint",CHECKPOINT_DIR,"--batch-size",BATCH_SIZE,"--output",RUN_DIR / "test_metrics.json"],
    enabled=RUN_TEST,log_path=RUN_DIR / "test.log")
if RUN_TEST:display(read_json(RUN_DIR / "test_metrics.json"))


## Mang kết quả về máy
Giữ **cả thư mục checkpoint** (đặc biệt `tang0_last.pt` và tokenizer) để resume.
`last` gồm model, optimizer, RNG, vị trí batch và best đã xác thực nếu có; chưa hết epoch vẫn có thể xuất.
`best` + cấu hình/tokenizer dùng cho suy luận. Nếu bị ngắt giữa validation thì lần sau chạy lại validation, không học lại các batch đã lưu.
Checkpoint trên ổ đĩa phiên làm việc chưa phải bản sao bền vững: cần lưu/tải output trước khi phiên bị xóa. Tự lưu định kỳ không đảm bảo Kaggle giữ output khi phiên bị hủy.
Ô dưới chỉ đóng gói khi chủ động bật; không upload tự động. Trên Kaggle cần lưu phiên bản có output trước khi kết thúc phiên.

In [ ]:
EXPORT_ZIP=False
if EXPORT_ZIP:
    import shutil
    if not (CHECKPOINT_DIR / "tang0_last.pt").is_file():
        require_checkpoint(CHECKPOINT_DIR,0)
    archive=shutil.make_archive(str(RUN_DIR / "tang0_checkpoint"),"zip",root_dir=CHECKPOINT_DIR)
    print("Đã đóng gói:",archive)


## Bước tiếp theo
Đặt artifact hoàn chỉnh vào `checkpoints/tangN/` khi triển khai pipeline mặc định, hoặc truyền đường dẫn checkpoint khi đánh giá.
Sau khi train đủ 0/3/4, còn fit TF-IDF và fusion trên validation theo `docs/PIPELINE.md`.
Tầng 5 không tham gia huấn luyện và không sửa nhãn. Notebook đã hỗ trợ resume; chưa triển khai mixed precision hoặc multi-GPU. Chọn GPU T4 ×2 không làm vòng lặp hiện tại tự dùng cả hai GPU.